# Power plant analysis

This notebook contains an analysis focused on finding potential area locations to obtain air quality data from OpenAQ stations. For this, we make use of the Global Power Plant database, obtained from the World Resources Institute. This dataset contains a list of power plants along with information such as geolocation, capacity, fuel types, yearly electricity generation, amongst others.

Afterwards, we juxtapose this data with other data sources to better narrow down the final areas. 

## Setup

In [ ]:
from dotenv import load_dotenv
import duckdb
import json
import os
import numpy as np
import polars as pl
import plotly.graph_objects as go
import requests
import time
import warnings

from sklearn.metrics import silhouette_score

from quoi.viz import setup_chart_template, bold_text, build_subtitle, add_titles

load_dotenv()

setup_chart_template(from_notebook=True)

power_plant_filepath = os.path.join(os.getenv('POWER_PLANT_DATA_FILEPATH'), 'global_power_plant_database.csv')
volcano_filepath = os.path.join(os.getenv('VOLCANO_DATA_FILEPATH'), 'volcanoes_database.tsv')
OPEN_AQ_API_KEY = os.getenv('OPEN_AQ_API_KEY')

DELAY_S = 0.25

In [ ]:
conn = duckdb.connect('energy_air_quality.db')

# Import power plant data
_ = conn.execute("CREATE TABLE IF NOT EXISTS power_plants as SELECT * FROM '{}'".format(power_plant_filepath))

# Import volcano data
# Columns are renamed to snake_case prior to importing to SQL
# Note: this is not the original filename, it has been renamed for convenience
_df = pl.read_csv(volcano_filepath, separator="\t")
_df = _df.drop('Search Parameters').tail(-1)
_df = _df.rename({col: col.lower().replace(' ', '_').replace('(', '').replace(')', '') for col in _df.columns})
_ = conn.execute("CREATE TABLE IF NOT EXISTS volcanoes as SELECT * FROM _df")

# For calculating distances
_ = conn.execute("""
INSTALL spatial;
LOAD spatial;
""")

del _df

In [ ]:
MAP_COLORS = {
    'Coal': 'black',
    'Hydro': 'blue',
    'Solar': 'orange',
    'Wind': 'green',
    'Nuclear': 'red',
    'Wave and Tidal': 'lightblue',
    'Oil': 'darkgrey',
    'Gas': 'brown',
    'Waste': 'purple'
}

POLLUTANT_LEVELS = dict(
    high=('Coal', 'Oil', 'Gas'),
    low_medium=('Waste', 'Biomass', 'Geothermal', 'Hydro'),
    least=('Nuclear', 'Solar', 'Wind')
)

## Overview

There are 36 columns in the Global Power Plant database, but we're mostly interested in the following:
 - `country` and `country_long`: country alpha-3 code and name, respectively;
 - `latitude` and `longitude`;
 - `primary_fuel`: the main energy source used by the power plant.

Other columns such as `name` and `capacity_mw` are also used on occasion. We do not concern with power plant scale as much as it's fuel type, however.

Regarding a given power plant's geolocation, coordinates are provided with a maximum of 4 decimals. According to Wikipedia's article on [Decimal degrees](https://en.wikipedia.org/wiki/Decimal_degrees), 4 decimals allows us to narrow down a location to an area of 4 to 11 meters, depending on latitude.

### Worldwide view

In [ ]:
_query = """
SELECT gppd_idnr, country_long, latitude, longitude, name, primary_fuel, capacity_mw
FROM power_plants
ORDER BY primary_fuel
"""

df_map = conn.sql(_query).pl()
fig = go.Figure()



for f, group in df_map.group_by('primary_fuel', maintain_order=True):
    _name = f[0]
    fig.add_trace(go.Scattermap(lat=group['latitude'],
                                lon=group['longitude'],
                                name=_name,
                                marker=dict(color=MAP_COLORS.get(_name),
                                            size=group['capacity_mw'] / 350,
                                            sizemin=2),
                                customdata=np.stack((group['capacity_mw'], group['name']), axis=-1)))

fig.update_layout(map=dict(zoom=1),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

add_titles(fig,
           title='Distribution of power plants by location, fuel type and capacity',
           subtitle='Entry size represents capacity',
           legend_title='Fuel type')

fig.update_traces(hovertemplate='<br>'.join([
    '%{lat}˚, %{lon}˚',
    '<b>Name:</b> %{customdata[1]}',
    '<b>Fuel type:</b> %{fullData.name}',
    '<b>Capacity:</b> %{customdata[0]} MW',
    '<extra></extra>'
]))

fig.layout.template.layout.legend = None
fig.show()

### Fuel distribution

We can bin power plant fuel types by multiple metrics:
 - Capacity (we opted for median here);
 - Total power plants;
 - Total unique countries.

In [ ]:
_query = """
SELECT
    primary_fuel,
    COUNT(*) AS count,
    COUNT(DISTINCT country) AS total_countries,
    QUANTILE_CONT(capacity_mw, 0.5) AS capacity_q50,
    QUANTILE_CONT(capacity_mw, 0.95) AS capacity_q95,
    STDDEV(capacity_mw) AS std_capacity
FROM power_plants
GROUP BY primary_fuel
ORDER BY count DESC
"""

df = conn.sql(_query).pl()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['capacity_q50'],
                         y=df['count'],
                         marker=dict(size=df['total_countries'],
                                     sizemin=5),
                         customdata=df['primary_fuel'],
                         hovertemplate='<br>'.join([
                             '%{customdata}',
                             '<b>Median capacity (MW):</b> %{x}',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Unique countries:</b> %{marker.size}',
                             '<extra></extra>'
                         ]),
                         mode='markers'))

fig.update_layout(xaxis_type='log')
add_titles(fig,
           title='Distribution of power plant fuel types by median capacity, total counts and usage per country',
           subtitle='Marker size represents total countries using a given fuel type',
           x_title='Median capacity (MW)',
           y_title='Total power plants')


fig.show()

The entries with smaller sizes and / or closer to the bottom left corner of the chart will have smaller impact when calculating per country fuel type entropy. The ones occurring in less than 10% of countries (around 17) will be excluded outright.

We can also calculate geographic densities for the most common fuel types. This will help us decide on OpenAQ stations later:

In [ ]:
fig = go.Figure()

# Based on results from the previous chart, we select only fuel types with +50 unique countries
_selected_types = ('Solar', 'Oil', 'Hydro', 'Wind', 'Gas', 'Coal')

for t in _selected_types:
    _curr_view = df_map.filter(pl.col('primary_fuel') == t)
    fig.add_trace(go.Densitymap(lat=_curr_view['latitude'],
                                lon=_curr_view['longitude'],
                                name=t,
                                opacity=0.5,
                                colorscale=[[0, '#ffffff'], [1, MAP_COLORS[t]]],
                                radius=10))

fig.update_traces(showscale=False,
                  hovertemplate='<br>'.join([
                      '%{lat}˚, %{lon}˚',
                      '<b>Fuel type:</b> %{fullData.name}'
                      '<extra></extra>'
                  ]))

fig.update_layout(title=bold_text('Distribution of power plants fuel type density by location') + build_subtitle('Only fuel types in at least 50 countries are shown'),
                  map=dict(zoom=1),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

fig.show()

Alternatively, we can bin all fuel types into different types of pollutant levels. We bin these according to how many greenhouse gas emissions each energy source registers. For this, we rely on [Our World in Data](https://ourworldindata.org/safest-sources-of-energy), which measures the greenhouse gas emissions of many energy sources. Based on this and other information found online (for other fuel types such as Waste and Geothermal), we get the following mapping:

- `High pollutants`: Coal, Oil, Gas;
- `Low / Medium pollutants`: Waste, Biomass, Geothermal, Hydro;
- `Least pollutant`: Nuclear, Solar, Wind.

Solar energy generation generates greenhouse gas emissions [during production](https://en.wikipedia.org/wiki/Environmental_impact_of_electricity_generation#Solar_power). Since it's difficult to factor this into a power plant based analysis, we have considered only operation emissions and placed this fuel type into `Least pollutant`.

In [ ]:
fig = go.Figure()

for p, fuels in POLLUTANT_LEVELS.items():
    color = 'red' if p == 'high' else ('orange' if p == 'low_medium' else 'green')
    _curr_view = df_map.filter(pl.col('primary_fuel').is_in(fuels))
    fig.add_trace(go.Densitymap(lat=_curr_view['latitude'],
                                lon=_curr_view['longitude'],
                                name=p.capitalize().replace('_', ' '),
                                customdata=_curr_view['primary_fuel'],
                                opacity=0.5,
                                colorscale=[[0, '#ffffff'], [1, color]],
                                radius=10))

fig.update_traces(showscale=False,
                  hovertemplate='<br>'.join([
                      '%{lat}˚, %{lon}˚',
                      '<b>Pollutant level:</b> %{fullData.name}',
                      '<b>Fuel type:</b> %{customdata}',
                      '<extra></extra>'
                  ]))

fig.update_layout(title=bold_text('Distribution of power plants fuel type density by location') + build_subtitle('Only fuel types in at least 50 countries are shown'),
                  map=dict(zoom=1),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

fig.show()

This chart gives us a much clearer view of which OpenAQ stations could measure higher or lower levels of pollutant related emissions such as sulfur oxides (SOx) and nitrogen oxides (NOx). Germany looks like a good candidate, as westernmost parts of the country (such as those in North Rhine-Westphalia) display very high concentrations of highly pollutant fuels, the south (such as Bavaria) for medium / low ones and areas such as south Brandenburg that register almost exclusively lower pollutant fuel types. Other canditates would include Australia, South Korea, United States, Brazil, Chile, amongst others.

## Country-level analysis

We want a single country to obtain air quality data from. The reason why we keep this to a single location is to keep a small scale when fetching data, and to prevent unknown country-specific factors from altering results, such as policies. After selecting a country, we will choose 3 sub-areas, one where each fuel pollution level is highly concentrated. This will later allow us to obtain air quality readings on these sub-areas and compare readings such as sulfur oxides (SOx), nitrogen oxides (NOx) and particulate matter (PM), in order to see if they differ considerably.

Here's what we want:
 - A good distribution of power plant fuel types (or at the very least, fuel pollution levels);
 - Considerable amount of air quality stations on each different selected sub-area;
 - Little to no impact risk from other, non-captured factors (some of which are discussed below).

First, let's calculate the coefficient of variation (CV) for both a given country's fuel and pollution level distributions:

In [ ]:
_query = """
WITH
    -- This CTE is used to fill missing country - fuel type combinations in the results
    combinations AS (
        SELECT DISTINCT c.country_long, f.primary_fuel
        FROM (SELECT DISTINCT country_long FROM power_plants) c
        CROSS JOIN (SELECT DISTINCT primary_fuel FROM power_plants) f
    ),
    shares AS (
        SELECT
            country_long,
            primary_fuel,
            COUNT(*) AS count,
            COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY country_long) * 100 AS fuel_count_share,
            SUM(capacity_mw) AS total_capacity
            --SUM(capacity_mw) / (SUM(capacity_mw) OVER (PARTITION BY country_long)) * 100 AS capacity_share
        FROM power_plants
        GROUP BY country_long, primary_fuel
    )
SELECT
    s.country_long,
    s.primary_fuel,
    CASE
        WHEN s.primary_fuel IN ('Coal', 'Oil', 'Gas') THEN 'high'
        WHEN s.primary_fuel IN ('Waste', 'Biomass', 'Geothermal', 'Hydro') THEN 'low_medium'
        ELSE 'lowest'
    END AS pollution_level,
    COALESCE(d.count, 0) AS count,
    COALESCE(d.fuel_count_share, 0) AS fuel_count_share,
    COALESCE(d.total_capacity, 0) AS total_capacity
FROM combinations s LEFT JOIN shares d ON (s.country_long = d.country_long AND s.primary_fuel = d.primary_fuel)
ORDER BY country_long, primary_fuel, pollution_level
"""

df_country_dist = conn.sql(_query).pl()

df_country_summary = df_country_dist.group_by('country_long').agg(
    pl.col('count').sum(),
    (pl.col('fuel_count_share').std() / pl.col('fuel_count_share').mean()).alias('cv_fuel')
).sort('cv_fuel')

# We also want to have a measure of entropy at the pollution level
_df_level_summary = df_country_dist.group_by(['country_long', 'pollution_level']).agg(
    pl.col('count').sum()
).with_columns(
    ((pl.col('count') / pl.col('count').sum()).over(pl.col('country_long')) * 100).alias('pollution_level_share')
).group_by('country_long').agg(
    (pl.col('pollution_level_share').std() / pl.col('pollution_level_share').mean()).alias('cv_level')
)

df_country_summary = df_country_summary.join(_df_level_summary, on='country_long', how='full', coalesce=True)

del _df_level_summary

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_country_summary['cv_fuel'],
                         y=df_country_summary['cv_level'],
                         marker=dict(size=df_country_summary['count'].log(),
                                     sizemin=2, sizeref=0.25),
                         customdata=np.stack((df_country_summary['country_long'], df_country_summary['count']), axis=-1),
                         hovertemplate='<br>'.join([
                             '%{customdata[0]}',
                             '<b>%{xaxis.title.text}:</b> %{x}',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Total power plants:</b> %{customdata[1]}',
                             '<extra></extra>'
                         ]),
                         mode='markers'))

add_titles(fig,
           title='Distribution of countries by power plant fuel and pollution level variety',
           subtitle='Marker size represents total power plants for a given country (logarithmic)',
           x_title='CV fuel type',
           y_title='CV pollution level')

fig.show()

Since pollution level is defined based on fuel types, it comes as little surprise that there will be a correlation between the two metric's results. It is pretty much impossible to have low fuel type variety, but high pollution level variety, and vice-versa.

The problem with this approach is that even though we are trying to get a sense of variety when it comes to fuel usage and pollution levels, **we are disregarding geographical distributions**. So if a country has all sorts of power plants using different fuel types, but they're all evenly distributed across the map, their coefficient of variation values will be low, despite the fact that we don't want to consider these entries as they may result in consistent air quality readings in the results. We want both diverse fuel usage, but also high clusterability across different pollution levels. This differs from standard clusterability metrics such as Hopkins statistic, since we have known classes, and want to instead highlight countries where these are more isolated from each other.

There are many methods to measure this, and here we will rely on the Silhouette Score, a method often used to measure how well defined the resulting clusters from unsupervised methods are. Results will range from -1 (samples assigned to the wrong cluster, in this case, it could represent fuel types closer to a cluster of a different fuel pollution type than its own) to 1 (well defined clusters).

In [ ]:
pollutant_mapping = {item: key for key, item_list in POLLUTANT_LEVELS.items() for item in item_list}
_res = []

df_cluster = df_map.with_columns(
    pl.col('primary_fuel').replace(pollutant_mapping).alias('pollution_level')
).filter(pl.col('pollution_level').is_in(POLLUTANT_LEVELS.keys()))

for country, group in df_cluster.group_by('country_long', maintain_order=True):
    score = None
    if group['pollution_level'].n_unique() > 1 and group.shape[0] > 5:
        score = silhouette_score(group[['latitude', 'longitude']], group['pollution_level'])
    
    new_row = {'country_long': country[0], 'silhouette_score': score}
    # Add pollution level counts to summary df
    counts = {key: val[0] for key, val in group['pollution_level'].value_counts().transpose(column_names='pollution_level').to_dict(as_series=False).items()}
    new_row = new_row | counts
    _res.append(new_row)

df_cluster_scores = pl.DataFrame(_res).fill_null(0)
df_cluster_scores = df_cluster_scores.join(df_country_summary, on='country_long', how='left', coalesce=True)

del _res, new_row, counts

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_cluster_scores['silhouette_score'],
                         y=df_cluster_scores['cv_level'],
                         marker=dict(size=df_cluster_scores['count'].log(),
                                     sizemin=2, sizeref=0.25),
                         customdata=np.stack((df_cluster_scores['country_long'], df_cluster_scores['count']), axis=-1),
                         hovertemplate='<br>'.join([
                             '%{customdata[0]}',
                             '<b>%{xaxis.title.text}:</b> %{x}',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Total power plants:</b> %{customdata[1]}',
                             '<extra></extra>'
                         ]),
                         mode='markers'))

add_titles(fig,
           title='Distribution of countries by pollution level variety and cluster scores',
           subtitle='Marker size represents total power plants for a given country (logarithmic)',
           x_title='Silhouette Score',
           y_title='CV pollution level')

fig.show()

The more power plants a country has, the closer it is to a Silhouette Score of 0, meaning the clusters defined by each power plant fuel type's pollution level are overlapping. While the connection to the coefficient of variation for pollution levels isn't as strong, it appears to concentrate on lower levels (below 1), meaning the standard deviation is not greater than the mean, and there's not as much of a heavy tail on a given location's distribution. We want this value as low as possible, for higher consistency / balance.

This means we want big entries on the bottom right corner. Unfortunately, no such entries exist. However, this method was not implemented to automatically select the best location, but rather to highlight potential candidates, and looking at the bigger entries closer to the bottom right corner, we see entries already highlighted previously such as Chile, Brazil, Germany, as well as other locations such as China, Mexico, Poland, India, amongst others. When compromising on locations closer to this bottom right corner, we prioritize maintaining the coefficient of variation low instead of the Silhouette Score, for reasons explained in the note below.

<b style='color: tomato'>Note:</b> There is a problem that this approach did not consider. When we provided cluster distributions to the Silhouette Score method, we did not filter out power plants that are in remote regions. This is very likely why countries such as Portugal and France have such low Silhouette Scores. In Portugal for example, power plants in the Azores and Madeira archipelagos severely impact cluster means.


### Other potential factors

Now, there are some other things we should keep in mind. We previously said we want to measure power plant fuel usage impact by comparing sulfur oxides (SOx), nitrogen oxides (NOx) and particulate matter (PM) readings from nearby OpenAQ stations, but other factors could lead to an increase in these readings as well. SOx readings are also impacted by volcanoes and geothermal activity. For NOx, vehicles could also lead to increases, meaning locations with high population density such as large cities could be a factor as well. Finally, particulate matter (PM) readings are impacted by the local weather, forest fires, vehicle activity, amongst (many) others.

The second factor, large cities, is somewhat mitigated by the fact that power plant data goes as far as 2021, a year with severly reduced city activity due to the COVID-19 pandemic, so if we obtain data from OpenAQ from the same time interval, this bias _should_ not be as high as it could be, but we should not rely exclusively on this assumption, and select locations far away from densely populated areas just in case.


#### Volcanic activity

<span style='color: tomato'><b>Disclaimer:</b> This section was written by someone with very limited knowledge on Volcanic activity. Please take this with a grain of salt.</span>

When it comes to volcanoes and other geothermal activity distributions, we will have to rely on another data source to juxtapose data and further filter out candidates. We will also have to keep in mind the area of effect of vulcanic activity. For the purposes of this analysis, since we only consider volcanic activity as a selection criteria, we will consider volcanoes with either a recorded eruption since the 20th century, or entries logged as `Fumarolic`, `Anthropocene` and `Holocene`. Having said that, a deep dive on air quality readings in locations close to different levels of volcanic activity would make for interesting future work.

Volcanoes also lead to increases in PM readings, in part due to SO2 reacting with water vapor and forming sulfuric acid, which in turn react with ammonia to form major contributors of PM2.5 such as ammonium sulfate.

When searching online, a distance of 10 km from the source was pointed out multiple times as a threshold where components such as SO2 usually register considerable drops in air quality readings. PM readings would depend on particle diameter: smaller particles travel farther. For this component, having a single threshold is more difficult, as it can range from around 10 km, to multiple hundreds. Here, we will assume 50 km.

In [ ]:
_query = """
SELECT
    p.gppd_idnr,
    power_plant_lon,
    power_plant_lat,
    v.volcano_name,
    volcano_lon,
    volcano_lat,
    ST_Distance_Spheroid(p.coord, v.coord) AS distance
FROM (
    SELECT 
        gppd_idnr, 
        longitude AS power_plant_lon,
        latitude AS power_plant_lat,
        ST_Point(latitude, longitude) AS coord
    FROM power_plants
) p
JOIN (
    SELECT
        volcano_name,
        longitude AS volcano_lon,
        latitude AS volcano_lat,
        ST_Point(latitude, longitude) AS coord
    FROM volcanoes
    WHERE 
        last_known_eruption IN ('D1', 'D2')
        OR status = 'Fumarolic'
        OR status ILIKE '%Holocene%'
        OR status ILIKE '%Anthropocene%'
) v
ON ST_DWithin_Spheroid(p.coord, v.coord, 50000)
"""

df_close_matches = conn.sql(_query).pl()

The following power plants were found to be close to these volcanoes:

In [ ]:
df_matched_power_plants = df_close_matches[['gppd_idnr', 'power_plant_lon', 'power_plant_lat']].unique()
df_matched_volcanoes = df_close_matches[['volcano_name', 'volcano_lon', 'volcano_lat']].unique()

fig = go.Figure()

fig.add_trace(go.Scattermap(lat=df_matched_power_plants['power_plant_lat'],
                            lon=df_matched_power_plants['power_plant_lon'],
                            customdata=df_matched_power_plants['gppd_idnr'],
                            name='Power plants'))

fig.add_trace(go.Scattermap(lat=df_matched_volcanoes['volcano_lat'],
                            lon=df_matched_volcanoes['volcano_lon'],
                            customdata=df_matched_volcanoes['volcano_name'],
                            name='Volcanoes'))

fig.update_layout(map=dict(zoom=1),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

fig.update_traces(hovertemplate='<br>'.join([
    '%{lat}˚, %{lon}˚',
    '<b>Entry:</b> %{customdata}'
    '<extra></extra>'
]))

add_titles(fig,
           title='Power plants close to volcanoes',
           subtitle='These entries are considered close if they are within 50km of each other')

fig.layout.template.layout.legend = None
fig.show()

Locations such as Central America, Chile, Indonesia, Philippines, Japan, Iceland, Italy and Turkey are considerably impacted and cannot be considered. If we were to repeat the above process but without power plants close to any volcano:

In [ ]:
_query = """
WITH
    -- This CTE is used to fill missing country - fuel type combinations in the results
    combinations AS (
        SELECT DISTINCT c.country_long, f.primary_fuel
        FROM (SELECT DISTINCT country_long FROM power_plants) c
        CROSS JOIN (SELECT DISTINCT primary_fuel FROM power_plants) f
    ),
    shares AS (
        SELECT
            country_long,
            primary_fuel,
            COUNT(*) AS count,
            COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY country_long) * 100 AS fuel_count_share,
            SUM(capacity_mw) AS total_capacity
            --SUM(capacity_mw) / (SUM(capacity_mw) OVER (PARTITION BY country_long)) * 100 AS capacity_share
        FROM power_plants
        WHERE
            gppd_idnr NOT IN $matches
        GROUP BY country_long, primary_fuel
    )
SELECT
    s.country_long,
    s.primary_fuel,
    CASE
        WHEN s.primary_fuel IN ('Coal', 'Oil', 'Gas') THEN 'high'
        WHEN s.primary_fuel IN ('Waste', 'Biomass', 'Geothermal', 'Hydro') THEN 'low_medium'
        ELSE 'lowest'
    END AS pollution_level,
    COALESCE(d.count, 0) AS count,
    COALESCE(d.fuel_count_share, 0) AS fuel_count_share,
    COALESCE(d.total_capacity, 0) AS total_capacity
FROM combinations s LEFT JOIN shares d ON (s.country_long = d.country_long AND s.primary_fuel = d.primary_fuel)
ORDER BY country_long, primary_fuel, pollution_level
"""

df_country_dist_no_matches = conn.sql(_query, params={'matches': df_matched_power_plants['gppd_idnr'].to_list()}).pl()

df_country_summary_no_matches = df_country_dist_no_matches.group_by('country_long').agg(
    pl.col('count').sum(),
    (pl.col('fuel_count_share').std() / pl.col('fuel_count_share').mean()).alias('cv_fuel')
).sort('cv_fuel')

# We also want to have a measure of entropy at the pollution level
_df_level_summary_no_matches = df_country_dist_no_matches.group_by(['country_long', 'pollution_level']).agg(
    pl.col('count').sum()
).with_columns(
    ((pl.col('count') / pl.col('count').sum()).over(pl.col('country_long')) * 100).alias('pollution_level_share')
).group_by('country_long').agg(
    (pl.col('pollution_level_share').std() / pl.col('pollution_level_share').mean()).alias('cv_level')
)

df_country_summary_no_matches = df_country_summary_no_matches.join(_df_level_summary_no_matches, on='country_long', how='full', coalesce=True)

del _df_level_summary_no_matches

In [ ]:
df_map_no_match = df_map.filter(~pl.col('gppd_idnr').is_in(df_matched_power_plants['gppd_idnr'].to_list()))
_res = []

df_cluster = df_map_no_match.with_columns(
    pl.col('primary_fuel').replace(pollutant_mapping).alias('pollution_level')
).filter(pl.col('pollution_level').is_in(POLLUTANT_LEVELS.keys()))

for country, group in df_cluster.group_by('country_long', maintain_order=True):
    score = None
    if group['pollution_level'].n_unique() > 1 and group.shape[0] > 5:
        score = silhouette_score(group[['latitude', 'longitude']], group['pollution_level'])
    
    new_row = {'country_long': country[0], 'silhouette_score': score}
    # Add pollution level counts to summary df
    counts = {key: val[0] for key, val in group['pollution_level'].value_counts().transpose(column_names='pollution_level').to_dict(as_series=False).items()}
    new_row = new_row | counts
    _res.append(new_row)

df_cluster_scores_no_match = pl.DataFrame(_res).fill_null(0)
df_cluster_scores_no_match = df_cluster_scores_no_match.join(df_country_summary_no_matches, on='country_long', how='left', coalesce=True)

del _res, new_row, counts

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df_cluster_scores_no_match['silhouette_score'],
                         y=df_cluster_scores_no_match['cv_level'],
                         marker=dict(size=df_cluster_scores_no_match['count'].log(),
                                     sizemin=2, sizeref=0.25),
                         customdata=np.stack((df_cluster_scores_no_match['country_long'], df_cluster_scores_no_match['count']), axis=-1),
                         hovertemplate='<br>'.join([
                             '%{customdata[0]}',
                             '<b>%{xaxis.title.text}:</b> %{x}',
                             '<b>%{yaxis.title.text}:</b> %{y}',
                             '<b>Total power plants:</b> %{customdata[1]}',
                             '<extra></extra>'
                         ]),
                         mode='markers'))

add_titles(fig,
           title='Distribution of countries by pollution level variety and cluster scores',
           subtitle='Marker size represents total power plants for a given country (logarithmic)',
           x_title='Silhouette Score',
           y_title='CV pollution level')

fig.show()

Some countries remain the same, the ones we highlighted previously register a considerable drop in count, and changes in silhouette scores and fuel coefficient of variation.

This last step requires manually checking the big entries on the lower left quadrant on the map above. There are multiple good options, but we opt for Germany, as there is a clear separation between zones with more polluting power plants, some with moderate activity, some with lower, and ideally more isolated stations without any power plants close by.

We now turn to OpenAQ. We want a good sample of stations around these 4 selected areas:

In [ ]:
def build_bounding_box(lat_l, lat_h, lon_l, lon_h, close_loop=True):
    lat = [lat_h, lat_l, lat_l, lat_h]
    lon = [lon_h, lon_h, lon_l, lon_l]

    if close_loop:
        lat.append(lat_h)
        lon.append(lon_h)

    return {'lat': lat, 'lon': lon}

# Selected areas
# They are declared by a high and low latitude and longitude coordinate set
SELECTED_AREAS = {
    '1': {'lat_h': 51.7, 'lat_l': 50.7, 'lon_h': 8, 'lon_l': 6.4},
    '2': {'lat_h': 51.9, 'lat_l': 51.1, 'lon_h': 14.4, 'lon_l': 12.3},
    '3': {'lat_h': 48.9, 'lat_l': 48.1, 'lon_h': 12.8, 'lon_l': 11.5},
    '4': {'lat_h': 53.4, 'lat_l': 52.5, 'lon_h': 11.2, 'lon_l': 9},
}

In [ ]:
fig = go.Figure()

for p, fuels in POLLUTANT_LEVELS.items():
    color = 'red' if p == 'high' else ('orange' if p == 'low_medium' else 'green')
    _curr_view = df_map.filter((pl.col('primary_fuel').is_in(fuels)) & (pl.col('country_long') == 'Germany'))
    fig.add_trace(go.Scattermap(lat=_curr_view['latitude'],
                                lon=_curr_view['longitude'],
                                marker=dict(color=color),
                                name=p.capitalize().replace('_', ' '),
                                customdata=_curr_view['primary_fuel']))

for k, coords in SELECTED_AREAS.items():
    _kwargs = build_bounding_box(**coords)
    fig.add_trace(go.Scattermap(mode='lines',
                                fill='toself',
                                showlegend=False,
                                **_kwargs))

fig.update_traces(hovertemplate='<br>'.join([
                      '%{lat}˚, %{lon}˚',
                      '<b>Pollutant level:</b> %{fullData.name}',
                      '<b>Fuel type:</b> %{customdata}',
                      '<extra></extra>'
                  ]))

fig.update_layout(title=bold_text('Selected locations for air quality readings in Germany juxtaposed with power plant fuel pollution levels'),
                  map=dict(zoom=5, center={"lat": 51.9, "lon": 9.7}),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

fig.show()

Ideally we would have at least a handful of stations in each selected area to obtain readings from and then obtain aggregations, but we are dependent on OpenAQ station availability. So let's check which stations are available for each selected area:

In [ ]:
OPEN_AQ_API_URL = 'https://api.openaq.org/v3/locations'
HEADERS = {'X-API-Key': OPEN_AQ_API_KEY}

lines = []

for area_id, area in SELECTED_AREAS.items():
    try:
        area = SELECTED_AREAS[area_id]
        PARAMS = {
            # For some reason, providing this alongside `countries_id=50` results in a 5xx error
            'iso': 'DE',
            'bbox': ','.join([str(el) for el in [area['lon_l'], area['lat_l'], area['lon_h'], area['lat_h']]])
        }
        
        res = requests.get(OPEN_AQ_API_URL, headers=HEADERS, params=PARAMS)
        # Keep API calls to at most 4 rps
        time.sleep(DELAY_S)
        res.raise_for_status()
    
        res_json = res.json()
        entries_skipped = []
        try:
            for entry in res_json['results']:
                new_row = {
                    'area': int(area_id),
                    'id': entry['id'],
                    'name': entry['name'],
                    'sensor_units': [s['name'] for s in entry['sensors']],
                    'lat': entry['coordinates']['latitude'],
                    'lon': entry['coordinates']['longitude'],
                    'licenses': [(l['name'], l['attribution']['name']) for l in entry['licenses']],
                    'dt_start': entry['datetimeFirst']['utc'],
                    'dt_end': entry['datetimeLast']['utc'],
                }
        
                lines.append(new_row)
        except TypeError:
            entries_skipped.append(entry['id'])

        if len(entries_skipped) > 0:
            warnings.warn(f'Area {area_id} - {len(entries_skipped)} stations were skipped due to parsing errors: {entries_skipped}')
    
    except requests.exceptions.HTTPError:
        print(f'Area {area_id} API call returned {res.status_code}: {res.text}')

df_stations = pl.DataFrame(lines)
df_stations = df_stations.with_columns(
    pl.col('dt_start').str.to_datetime(time_zone='UTC'),
    pl.col('dt_end').str.to_datetime(time_zone='UTC'),
)

Let's get a brief summary of how many stations we got for each area and how many have data for 2021 and 2022:

In [ ]:
df_stations.group_by('area').agg(
    pl.col('id').n_unique().alias('total_stations'),
    ((pl.col('dt_start').dt.year() < 2021) & (pl.col('dt_end').dt.year() > 2022)).sum().alias('total_within_timerange')
).sort('area')

Unfortunately, there aren't many stations to provide us data, at least for the specified time interval. In order to get more entries, we would either need to adjust the time interval or change the locations we are obtaining station data from.

We will also have to forgo area 4, even after increasing the total area coverage considerably, we are left with too few stations to obtain data from. We will select 5 stations from each of the other 3 areas and obtain data from these over time.

In [ ]:
SELECTED_STATIONS = [
    3180, 3187, 3078, 3158, 3068, # Area 1
    2935, 3915, 4701, 4751, 4752, # Area 2
    2669, 3093, 3099, 3202, 4557  # Area 3
]

In [ ]:
fig = go.Figure()

_temp = df_stations.filter((pl.col('id').is_in(SELECTED_STATIONS)))
fig.add_trace(go.Scattermap(lat=_temp['lat'],
                            lon=_temp['lon'],
                            marker=dict(size=10),
                            customdata=_temp['id']))


fig.update_traces(hovertemplate='<br>'.join([
                      '%{lat}˚, %{lon}˚',
                      '<b>Id:</b> %{customdata}'
                      '<extra></extra>'
                  ]))

fig.update_layout(title=bold_text('Final selected OpenAQ station locations'),
                  map=dict(zoom=5, center={"lat": 51.9, "lon": 9.7}),
                  height=900, width=1300,
                  margin=dict(b=10, l=20, r=10))

fig.show()

Area 3 (stations in the South, in Bavaria) have very few stations and most are within Munich, which is likely going to skew results due to it being in a dense urban area. Stations in area 2 (in the east, Saxony-Anhalt and Brandenburg) are organized into two concentrated poles, rather than being evenly distributed. For area 1 (west, North Rhine-Westphalia), since it is the area with the most stations to choose from, we aim to select stations more or less far apart and not within large cities.

Having obtained a list of stations to obtain data from, we can now conclude this analysis.

## Conclusions

In this analysis, we looked into locations whose power plant fuel usage was more organized into well defined clusters, by calculating the coefficient of variation (CV) of power plant locations within each country by the fuel pollution type, as well as the silhouette score of these same groups. We then filter power plants that were deemed too close (within 50km) to volcanoes which registered activity relatively recently, in order to avoid interference in readings such as SOx and PM2.5. Finally, we establish multiple areas within the country, differing by distribution of used power plant fuel pollution levels.

This analysis could be further improved by adding population density data as an enrichment source. It would provide us better filtering (avoid areas that are large cities and thus may have high emissons such as NO2), and also could be used to test if there was a correlation between population density and the distribution of OpenAQ stations across the world.

## Citations

Hannah Ritchie (2020) - “What are the safest and cleanest sources of energy?” Published online at OurWorldinData.org. Retrieved from: 'https://archive.ourworldindata.org/20260810-152248/safest-sources-of-energy.html' [Online Resource] (archived on August 10, 2026).